# 01 — Historical archive

The competition exposes a historical archive so you can experiment **offline**:
one gzip-JSONL file per `event_type` × `quarter`, each line a past event with its
disclosure. This notebook:

1. Browses the archive manifest (`GET /archive`)
2. Downloads and **caches** the files locally (`data/archive/`, gitignored)
3. Loads them into a pandas DataFrame
4. Inspects a single event and its disclosure

With a live key you pull the real archive; without one, you load the small bundled
sample so the mechanics are still clear.

> Download URLs are short-lived and signed. The helper refreshes an expired URL
> automatically (via `GET /archive/{event_type}/{quarter}`) when you pass it a
> client.

In [1]:
from pathlib import Path

from IPython.display import display

from examples import Client, load_config
from examples.archive import download_archive, load_archive
from examples.frames import manifest_frame

def _find_repo() -> Path:
    """Locate the `examples/` package root (has `data/sample`) from wherever this notebook runs.

    Checks cwd and its parents (works if the notebook lives inside examples/,
    e.g. examples/notebooks/), and cwd/examples (works if the notebook lives
    at the starter-modal/ root, one level above examples/).
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "sample").is_dir():
            return p
        if (p / "examples" / "data" / "sample").is_dir():
            return p / "examples"
    raise FileNotFoundError("Could not find examples/data/sample from cwd, its parents, or ./examples.")

REPO = _find_repo()
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache
SAMPLE_DIR = REPO / "data" / "sample"  # tiny bundled stand-in

config = load_config()
print("Mode:", "LIVE" if config.is_live else "SAMPLE (no EM_API_KEY — using bundled data)")

Mode: LIVE


## 1. Browse the manifest

Each row is one downloadable file — what event type and quarter it covers, how
many events it holds, and how big it is. (The signed URLs are omitted from this
view.)

In [2]:
manifest = None
if config.is_live:
    with Client.from_env() as client:
        manifest = client.archive_manifest()
    display(manifest_frame(manifest))
else:
    print("Sample mode: no live manifest. We'll load the bundled sample archive below.")

,event_type,quarter,events,size_mb,sealed,covers_from,covers_to
0,EARNINGS_RELEASE,2025Q4,1849,1.04,True,2025-10-09T10:00:00+00:00,2025-11-14T21:05:00+00:00
1,EARNINGS_RELEASE,2026Q1,2391,1.63,True,2026-01-06T11:30:00+00:00,2026-03-31T21:47:39+00:00
2,EARNINGS_RELEASE,2026Q2,2060,1.71,True,2026-04-01T00:00:00+00:00,2026-06-30T20:15:00+00:00
3,EARNINGS_RELEASE,2026Q3,2191,1.74,False,2026-07-01T10:30:00+00:00,2026-08-28T13:00:00+00:00


## 2. Download & cache

`download_archive` writes each file into `data/archive/`, **skipping** any file
already cached at the size the manifest reports — so re-running is cheap. Narrow
the pull with `only=`, e.g. `only=lambda f: f.quarter >= "2026Q1"`.

In [3]:
if config.is_live:
    with Client.from_env() as client:
        paths = download_archive(
            manifest,
            ARCHIVE_DIR,
            client=client,  # lets it refresh expired URLs
            # only=lambda f: f.event_type == "EARNINGS_RELEASE",
        )
    source = ARCHIVE_DIR
    print(f"{len(paths)} file(s) cached in {ARCHIVE_DIR}")
else:
    source = SAMPLE_DIR
    print(f"Sample mode: reading the bundled archive in {SAMPLE_DIR}")

skip     EARNINGS_RELEASE_2025Q4.jsonl.gz (cached, 1,036,733 bytes)
skip     EARNINGS_RELEASE_2026Q1.jsonl.gz (cached, 1,625,978 bytes)
skip     EARNINGS_RELEASE_2026Q2.jsonl.gz (cached, 1,710,368 bytes)
skip     EARNINGS_RELEASE_2026Q3.jsonl.gz (cached, 1,744,213 bytes)
4 file(s) cached in /Users/michaelmeier/starter-modal/examples/data/archive


## 3. Load into pandas

`load_archive` reads every `*.jsonl.gz` under a directory (or a single file) into
one DataFrame, adding `event_type` and `quarter` provenance columns from each
filename.

In [4]:
df = load_archive(source)
print(f"Loaded {len(df):,} events")
print("Columns:", list(df.columns))
df.head()

Loaded 8,491 events
Columns: ['quarter', 'event_id', 'event_type', 'timing_category', 'event_datetime', 'knowledge_cutoff', 'focal_assets', 'status', 'return_status', 'event_returns', 'returns_computed_at', 'metrics', 'baseline_predictions', 'disclosure']


,quarter,event_id,event_type,timing_category,event_datetime,knowledge_cutoff,focal_assets,status,return_status,event_returns,returns_computed_at,metrics,baseline_predictions,disclosure
0,2025Q4,ea_PEP_Q3_2025,EARNINGS_RELEASE,SCHEDULED,2025-10-09T10:00:00+00:00,2025-10-08T20:00:00+00:00,"[{'identifier_type': 'TICKER', 'identifier_val...",scored,ok,"{'PEP': {'window_start_date': '2025-10-08', 'r...",2026-07-10T22:18:51.501247+00:00,{'earnings_surprise': {'surprise': 0.000216076...,{'gemini/ea-explain-contemp-summary': {'PEP': ...,"{'schema_version': '1.0', 'event_id': 'ea_PEP_..."
1,2025Q4,ea_DAL_Q3_2025,EARNINGS_RELEASE,SCHEDULED,2025-10-09T10:30:00+00:00,2025-10-08T20:00:00+00:00,"[{'identifier_type': 'TICKER', 'identifier_val...",scored,ok,"{'DAL': {'window_start_date': '2025-10-08', 'r...",2026-07-10T22:18:51.948245+00:00,{'earnings_surprise': {'surprise': 0.002801120...,{'gemini/ea-explain-contemp-summary': {'DAL': ...,"{'schema_version': '1.0', 'event_id': 'ea_DAL_..."
2,2025Q4,ea_NEOG_Q1_2026,EARNINGS_RELEASE,SCHEDULED,2025-10-09T11:00:00+00:00,2025-10-08T20:00:00+00:00,"[{'identifier_type': 'TICKER', 'identifier_val...",scored,ok,"{'NEOG': {'window_start_date': '2025-10-08', '...",2026-07-10T22:19:25.241453+00:00,{'earnings_surprise': {'surprise': 0.001718213...,{'gemini/ea-explain-contemp-summary': {'NEOG':...,"{'schema_version': '1.0', 'event_id': 'ea_NEOG..."
3,2025Q4,ea_BYRN_Q3_2025,EARNINGS_RELEASE,SCHEDULED,2025-10-09T12:00:00+00:00,2025-10-08T20:00:00+00:00,"[{'identifier_type': 'TICKER', 'identifier_val...",scored,ok,"{'BYRN': {'window_start_date': '2025-10-08', '...",2026-07-10T22:19:29.712564+00:00,{'earnings_surprise': {'surprise': 0.001754385...,{'gemini/ea-explain-contemp-summary': {'BYRN':...,"{'schema_version': '1.0', 'event_id': 'ea_BYRN..."
4,2025Q4,ea_APOG_Q2_2026,EARNINGS_RELEASE,SCHEDULED,2025-10-09T20:00:00+00:00,2025-10-08T20:00:00+00:00,"[{'identifier_type': 'TICKER', 'identifier_val...",scored,ok,"{'APOG': {'window_start_date': '2025-10-09', '...",2026-07-10T22:17:53.134327+00:00,{'earnings_surprise': {'surprise': 0.003137065...,{'gemini/ea-explain-contemp-summary': {'APOG':...,"{'schema_version': '1.0', 'event_id': 'ea_APOG..."


## 4. Inspect one event

Pull a single row and read its focal assets and disclosure facts. (The bundled
sample inlines disclosures under a `disclosure` field; the live archive may carry
additional fields — inspect `df.columns` to see what a given dataset provides.)

In [5]:
row = df.iloc[0]

focal = row.get("focal_assets") if "focal_assets" in df.columns else None
tickers = [a["identifier_value"] for a in (focal or [])]
print(f"event_type={row['event_type']}  quarter={row['quarter']}  tickers={tickers}")
print(f"event_datetime={row.get('event_datetime')}\n")

disclosure = row.get("disclosure") if "disclosure" in df.columns else None
if isinstance(disclosure, dict):
    for item in disclosure.get("items", []):
        if item.get("kind") == "facts":
            print(f"facts from {item.get('source')!r}:")
            for fact in item["content"]:
                print("  -", fact)
else:
    print("No inlined 'disclosure' column here — check df.columns for this dataset's fields.")

event_type=EARNINGS_RELEASE  quarter=2025Q4  tickers=['PEP']
event_datetime=2025-10-09T10:00:00+00:00

facts from 'earnings_call':
  - PepsiCo's beverage business actually grew volume in Q3 when excluding the case pack water divestment, with Pepsi brand growing volume, net revenue, and market share.
  - Service levels for both food and beverage businesses are now at 97%-98% following earlier systems transition issues.
  - The Away From Home channel is growing 2-3x faster than the retail business in both U.S. and internationally.
  - PepsiCo expects to return to its long-term algorithm for net revenue growth within 2026, though timing may vary between Q3 and Q4.
  - The Frito-Lay food business in North America returned to growth in the last 4 weeks of the quarter, with management expecting close to flat organic sales in Q4.
  - International business returned to mid- to high mid-single-digit performance in September after a weaker summer impacted by weather.
  - Brazil delivered close t

## 5. Reproduce the baseline scoring regressions

`examples.scoring` ports the competition's **exact** scoring transform, so you can
(i) reproduce the leaderboard's numbers offline and (ii) use it in your own
analyses and optimizations. Scoring runs *within a period* (a quarter): the
realized one-day cumulative abnormal return (`car1`) of every outcome is
percentile-ranked into the dependent variable `y = r_abn`, and the earnings
surprise is percentile-ranked into the `s` regressor. Each reference baseline
already submitted a predicted percentile `ĝ`.

We reproduce **Table 3** of Koijen & Levy (WP) on the disseminated data: `y`
regressed on five specifications —

1. surprise only (`s`) — the naive benchmark,
2. Gemini only (`ĝ`),
3. GPT-5 nano only (`ĝ`),
4. surprise + Gemini,
5. surprise + GPT-5 nano.

This section needs the **live** archive (the bundled sample carries no returns or
baseline predictions). We use `2025Q4`, the sealed quarter behind the paper's
table. (The paper also reports bootstrapped R² intervals; we show point estimates
and standard errors.)

In [6]:
from examples.scoring import BASELINE_LABELS, add_percentiles, outcomes_frame

PERIOD = "2026Q3"

needed = {"event_returns", "metrics", "baseline_predictions"}
have_scoring_data = needed <= set(df.columns) and (df["quarter"] == PERIOD).any()

if have_scoring_data:
    # Percentiles are ranked WITHIN the period, so restrict to the quarter first.
    records = df[df["quarter"] == PERIOD].to_dict("records")
    scored = add_percentiles(outcomes_frame(records))
    models = list(BASELINE_LABELS.values())  # ["Gemini 2.5 Flash-Lite", "GPT-5 nano"]
    # The paper's single sample: outcomes with y, a surprise, and both baselines.
    fit_df = scored.dropna(subset=["y", "surprise_pct", *models]).copy()
    print(f"{PERIOD}: {len(scored):,} scored outcomes | fit sample N = {len(fit_df):,}")
else:
    print(f"Sample mode: section 5 needs the live archive for {PERIOD}. Skipping.")

2026Q3: 2,191 scored outcomes | fit sample N = 2,111


In [7]:
if have_scoring_data:
    import pandas as pd
    import statsmodels.api as sm
    from stargazer.stargazer import Stargazer

    gemini, gpt = models  # display names, in BASELINE_LABELS order
    S, G = "s (surprise pctile)", "ĝ (model pctile)"

    def fit(cols):
        # cols: {display_name -> source column}; shared names let stargazer stack rows.
        X = sm.add_constant(pd.DataFrame({name: fit_df[src] for name, src in cols.items()}))
        return sm.OLS(fit_df["y"].to_numpy(), X).fit()

    fits = [
        fit({S: "surprise_pct"}),  # (1) Baseline: surprise only
        fit({G: gemini}),  # (2) Gemini only
        fit({G: gpt}),  # (3) GPT-5 nano only
        fit({S: "surprise_pct", G: gemini}),  # (4) surprise + Gemini
        fit({S: "surprise_pct", G: gpt}),  # (5) surprise + GPT-5 nano
    ]

    star = Stargazer(fits)
    star.title(f"Transcript summaries — Koijen & Levy (WP) Table 3, reproduced on {PERIOD}")
    star.custom_columns(["Baseline", gemini, gpt, gemini, gpt], [1, 1, 1, 1, 1])
    star.covariate_order(["const", S, G])
    star.rename_covariates({"const": "Intercept"})
    star.dependent_variable_name("r_abn  (percentiled CAR1)")
    star.show_model_numbers(True)
    display(star)

## 6. Backtest: predict.py vs predict_2_2.py — does the Finnhub EPS-surprise signal help?

**Status: `predict.py` was promoted.** After this A/B showed the Finnhub
surprise signal improving `delta_r_squared` on a clean run, the file that used
to be `predict_finnhub.py` was renamed to `predict.py` (live), and the old
pre-Finnhub `predict.py` was archived as `predict_2_2.py` (not deployed, kept
for reference / this A/B). This section now runs `predict.py` (current live:
GLM-5.2 + rulebook + insider signal + Finnhub EPS surprise) against
`predict_2_2.py` (the same, minus the Finnhub signal) — same comparison as
before, roles swapped to match the new file names.

The archive has no `information_url`; its facts live inline under
`disclosure.items[].content`, so we rebuild an equivalent summary text from
there.

Results are cached to disk (`backtest_cache_v22.json` — a **fresh** file, not
the old `backtest_cache_finnhub.json`, whose `"predict"` key meant something
different pre-promotion and would silently mix code versions if reused), keyed
by `event_id`. Start with a small sample (`N_SAMPLE` below) before running a
full quarter — every event costs 2x real, billed GLM-5.2 calls (one per
variant), plus a Finnhub call per event for the `predict` side.

In [8]:
import sys
import time

period_df = df[df["quarter"] == PERIOD].copy()  # PERIOD/df/scored come from Section 5

REPO_ROOT = REPO.parent  # starter-modal/ — one level above examples/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
import predict  # noqa: E402  (live predictor -- GLM-5.2 + rulebook + insider + Finnhub surprise)
import predict_2_2  # noqa: E402  (archived pre-Finnhub snapshot, see its module docstring)

def facts_to_summary(disclosure: dict) -> str:
    """Rebuild an information_url-shaped summary string from an archive disclosure."""
    lines = []
    for item in disclosure.get("items", []):
        if item.get("kind") == "facts":
            lines.extend(item["content"])
    return "\n".join(f"- {f}" for f in lines)

print("ZAI_API_KEY set:", bool(os.environ.get("ZAI_API_KEY")))
print("FINNHUB_API_KEY set:", bool(os.environ.get("FINNHUB_API_KEY")))

ZAI_API_KEY set: True
FINNHUB_API_KEY set: True


In [9]:
# Quick peek: print the exact system+user prompt for just the first 2 events,
# for both predict.py and predict_2_2.py -- not the full N_SAMPLE run below,
# which would flood the output with ~200 prints. Overriding PROMPT_LOG_EVERY
# as a plain module attribute (not the env var) works fine here -- no
# re-import needed -- and resetting it after doesn't touch anything already
# printed.
predict.PROMPT_LOG_EVERY = 1
predict_2_2.PROMPT_LOG_EVERY = 1

for _, row in period_df.head(2).iterrows():
    ticker = row["focal_assets"][0]["identifier_value"]
    summary_text = facts_to_summary(row["disclosure"])
    predict._ask_llm(
        summary={"summary": summary_text},
        ticker=ticker,
        event_type=row["event_type"],
        as_of=row["knowledge_cutoff"],
    )
    predict_2_2._ask_llm(
        summary={"summary": summary_text},
        ticker=ticker,
        event_type=row["event_type"],
        as_of=row["knowledge_cutoff"],
    )

# Reset so the full backtest run below doesn't print every prompt too.
predict.PROMPT_LOG_EVERY = 100
predict_2_2.PROMPT_LOG_EVERY = 100


==================== predict.py -- SYSTEM PROMPT (call #1) ====================
You are a senior equity analyst predicting how a stock will react to an event.

Predict a single percentile in [0, 1] for how the focal asset's next-day
abnormal return will rank across all of the quarter's event outcomes:
0 = the quarter's most negative reaction, 0.50 = median, 1 = its most positive.
The relevant return is the *unexpected*, market-adjusted return — a
great-but-fully-priced-in beat is not a top-decile event.

Calibration discipline:
- Long-run base rates: about 25% of events land "up" (>0.75), 50% "neutral"
  (0.25-0.75), 25% "down" (<0.25). Default toward 0.40-0.60 when signals are
  mixed or modest.
- Reserve values above 0.80 or below 0.20 for cases with unambiguous,
  multi-signal evidence. Do not exceed 0.90 or fall below 0.10 without
  overwhelming, lopsided evidence.
- Tone alone (confident vs hedging language) should move you no more than
  ~0.10 absent quantitative confirmation.



In [10]:
import json
import os
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Fresh cache for the post-promotion pairing (predict.py = now Finnhub-enhanced
# vs predict_2_2.py = archived pre-Finnhub) -- reusing backtest_cache_finnhub.json
# would mix pre-promotion "predict" results (the old non-Finnhub code) with
# post-promotion ones (the new Finnhub-enhanced code) under the same key.
CACHE_PATH = Path.cwd() / "backtest_cache_v22.json"
N_DRAWS = 1  # average this many independent draws per prediction -- smooths
# temperature=1 sampling noise, same fix predict.py already applies live
# (N_ENSEMBLE_DRAWS) -- both variants get it equally here, so it's not a
# confound between them, just less noise in the comparison overall.
MAX_WORKERS = 10  # outer (event-level) concurrency -- each event now fires up
# to N_DRAWS concurrent calls per model, so this stays low to keep total
# concurrent GLM-5.2 calls (MAX_WORKERS * N_DRAWS) in the same ballpark as
# the old single-draw MAX_WORKERS=10.

def load_cache(path: Path) -> dict:
    if not path.exists() or path.stat().st_size == 0:
        return {}
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        print(f"[WARN] {path.name} was empty/corrupt (likely an interrupted write) -- starting fresh.")
        return {}

def save_cache(path: Path, data: dict) -> None:
    # Write to a temp file and atomically replace -- a kernel interrupt mid-write
    # can no longer truncate the real cache file.
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2))
    os.replace(tmp, path)

cache = load_cache(CACHE_PATH)
cache_lock = threading.Lock()

N_SAMPLE = 200  # near-full quarter (772 events) -- keeps n_obs stable/comparable run to run,
# instead of bouncing around on a small, randomly-varying covered subset
sample_df = period_df.sample(n=min(N_SAMPLE, len(period_df)), random_state=0)

todo = [row for _, row in sample_df.iterrows() if row["event_id"] not in cache]
print(f"{len(sample_df) - len(todo)} already cached, {len(todo)} to run now")
print(f"Each event now costs 2 x {N_DRAWS} = {2 * N_DRAWS}x the LLM calls (predict.py + predict_2_2.py, {N_DRAWS} draws each, per ticker), plus one Finnhub call per event for predict.")

def _ensembled(draw_fn) -> float:
    """Average N_DRAWS concurrent calls to `draw_fn` into one percentile."""
    with ThreadPoolExecutor(max_workers=N_DRAWS) as pool:
        draws = list(pool.map(lambda _: draw_fn(), range(N_DRAWS)))
    return sum(draws) / len(draws)

def predict_one(row):
    summary_text = facts_to_summary(row["disclosure"])
    tickers = [a["identifier_value"] for a in row["focal_assets"]]

    preds = {"predict": {}, "predict_2_2": {}}
    for ticker in tickers:
        # BOTH variants' _ask_llm now return a Prediction (predicted_percentile +
        # rationale) -- extract .predicted_percentile consistently before averaging.
        # predict_2_2.py is the archived pre-Finnhub snapshot -- see its module
        # docstring. Both self-fetch insider_summary/surprise_signal here since
        # they're called directly (not through _ensembled_percentile) -- that's
        # the documented direct-call fallback path in both files.
        preds["predict"][ticker] = _ensembled(lambda: predict._ask_llm(
            summary={"summary": summary_text},
            ticker=ticker,
            event_type=row["event_type"],
            as_of=row["knowledge_cutoff"],
        ).predicted_percentile)
        preds["predict_2_2"][ticker] = _ensembled(lambda: predict_2_2._ask_llm(
            summary={"summary": summary_text},
            ticker=ticker,
            event_type=row["event_type"],
            as_of=row["knowledge_cutoff"],
        ).predicted_percentile)
    return row["event_id"], preds

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(predict_one, row): row["event_id"] for row in todo}
    done = 0
    failed = []
    for future in as_completed(futures):
        event_id = futures[future]  # known even if the call below raises
        try:
            event_id, preds = future.result()
        except Exception as e:
            # A single timeout/API error used to crash the whole cell here --
            # every other future still running in the background would finish
            # (the executor waits for them on exit) but never get cached, since
            # the loop had already exited. Now we just log and move on: this
            # event_id stays out of the cache, so it's automatically retried
            # the next time you run this cell (it'll show up in 'todo' again).
            print(f"[ERROR] event={event_id} failed: {e!r} -- skipping, not cached")
            failed.append(event_id)
            continue
        with cache_lock:
            cache[event_id] = preds
            save_cache(CACHE_PATH, cache)  # still resume-safe if interrupted mid-run
        done += 1
        if done % 100 == 0 or done == len(todo):
            print(f"{done}/{len(todo)} done")

print(f"{len(cache)} events predicted (cached in {CACHE_PATH.name})")
if failed:
    print(f"{len(failed)} event(s) failed and were skipped: {failed}")
    print("Re-run this cell to retry them -- they were never added to the cache.")

0 already cached, 200 to run now
Each event now costs 2 x 1 = 2x the LLM calls (predict.py + predict_2_2.py, 1 draws each, per ticker), plus one Finnhub call per event for predict.
[ERROR] event=ea_ORI_Q2_2026 failed: AttributeError("'float' object has no attribute 'predicted_percentile'") -- skipping, not cached
[ERROR] event=ea_PFBC_Q2_2026 failed: AttributeError("'float' object has no attribute 'predicted_percentile'") -- skipping, not cached
[ERROR] event=ea_NRC_Q2_2026 failed: AttributeError("'float' object has no attribute 'predicted_percentile'") -- skipping, not cached
[ERROR] event=ea_ALG_Q2_2026 failed: AttributeError("'float' object has no attribute 'predicted_percentile'") -- skipping, not cached
[ERROR] event=ea_ELAN_Q2_2026 failed: AttributeError("'float' object has no attribute 'predicted_percentile'") -- skipping, not cached
[ERROR] event=ea_KE_Q4_2026 failed: AttributeError("'float' object has no attribute 'predicted_percentile'") -- skipping, not cached
[ERROR] event=

In [11]:
# examples.scoring.score_submission is the exact port of the contest's own
# scorer (Official Rules §5) — same metric family as the leaderboard, and
# directly comparable to the Gemini/GPT-5-nano numbers from Section 5.
from examples.scoring import score_submission

def score_model(model_key: str) -> dict:
    rows = [
        {"event_id": event_id, "identifier_value": ticker, "predicted": pct}
        for event_id, preds in cache.items()
        for ticker, pct in preds[model_key].items()
    ]
    pred_df = pd.DataFrame(rows)
    # Join on (event_id, identifier_value) — `scored` has one row per (event, asset),
    # so joining on event_id alone would mis-match events with multiple tickers.
    merged = scored.merge(pred_df, on=["event_id", "identifier_value"], how="left")
    covered = merged["predicted"].notna().sum()
    print(f"{covered} / {len(pred_df)} cached predictions matched to a scored outcome")
    return score_submission(merged, "predicted")

# A/B: predict.py (current live -- now includes the Finnhub EPS-surprise
# signal) vs predict_2_2.py (the archived pre-Finnhub snapshot) -- isolates
# whether that one signal moves delta_r_squared.
results = {}
for model_key, label in [("predict", "predict.py (GLM-5.2 + Finnhub EPS surprise, live)"), ("predict_2_2", "predict_2_2.py (GLM-5.2 baseline, archived)")]:
    print(f"=== {label} ===")
    results[model_key] = score_model(model_key)
    for key in ("n_obs", "r_squared", "r_squared_surprise", "delta_r_squared"):
        print(f"{key:>20}: {results[model_key][key]}")
    print()

# Sections 7/8 below plot/log a single `result` -- keep it pointed at predict.py
# so those cells run unchanged. `results["predict_2_2"]` holds the archived
# baseline's scores if you want to compare its numbers directly.
result = results["predict"]
for key in ("r_squared_imputed", "delta_r_squared_imputed", "imputed_event_count"):
    print(f"{key:>20}: {result[key]}")

=== predict.py (GLM-5.2 + Finnhub EPS surprise, live) ===


KeyError: 'event_id'

## 7. Compare against Table 3 baselines

Same PERIOD, same `y`. Baseline numbers are hardcoded from Section 5's Table 3
reproduction (N=1849, full quarter) — the model bars are `predict` and
`predict_2_2`'s `results` from Section 6, and are only sample-size-comparable
once `n_obs` approaches 1849.

In [ ]:
import matplotlib.pyplot as plt

# Reference numbers from Koijen & Levy (WP) Table 3 — N=1849, full quarter
labels = [
    "Surprise only\n(N=1849)",
    "Gemini\n(N=1849)",
    "GPT-5 nano\n(N=1849)",
    f"predict\n(N={results['predict']['n_obs']})",
    f"predict_2_2\n(N={results['predict_2_2']['n_obs']})",
]
r2_values = [0.049, 0.098, 0.072, results["predict"]["r_squared"], results["predict_2_2"]["r_squared"]]
delta_values = [
    0.0, 0.098 - 0.049, 0.072 - 0.049,
    results["predict"]["delta_r_squared"], results["predict_2_2"]["delta_r_squared"],
]
colors = ["#888888", "#4C72B0", "#DD8452", "#55A868", "#8172B2"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(labels, r2_values, color=colors)
axes[0].set_ylabel("R^2 (surprise + model, two-regressor)")
axes[0].set_title("R^2 vs. Table 3 baselines")
axes[0].tick_params(axis="x", labelrotation=15)

axes[1].bar(labels, delta_values, color=colors)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Delta R^2 over surprise-only")
axes[1].set_title("Incremental alpha over surprise")
axes[1].tick_params(axis="x", labelrotation=15)

plt.tight_layout()
plt.show()

print(
    f"Baselines: N=1849 (full quarter). predict: N={result['n_obs']} (ungefuellt) "
    "-- only directionally comparable until n_obs approaches 1849.\n"
    f"Official contest metric (imputed, full N=1849, {result['imputed_event_count']} mean-filled): "
    f"delta_r_squared_imputed = {result['delta_r_squared_imputed']:.5f}"
)

## 8. Log this run (tracks ranking-relevant metric over time)

The contest ranks on `delta_r_squared` over the **common sample** (Official
Rules §5 / imputed family) — every run here appends `predict`'s `result`
to `delta_r2_log.csv` so you can see whether changes to `predict.py`
(prompt, rulebook, added data sources) are actually moving that number,
instead of eyeballing one run at a time. Only `predict` is logged here;
`results["predict_2_2"]` is available in Section 6/7 if you want to track
that one too.

In [ ]:
import csv
from datetime import datetime, timezone

LOG_PATH = Path.cwd() / "delta_r2_log.csv"
RUN_NOTES = ""  # fill in before running, e.g. "added yfinance trend+vol, reasoning=high"

row = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "period": PERIOD,
    "n_obs": result["n_obs"],
    "r_squared": result["r_squared"],
    "r_squared_surprise": result["r_squared_surprise"],
    "delta_r_squared": result["delta_r_squared"],
    "r_squared_imputed": result["r_squared_imputed"],
    "delta_r_squared_imputed": result["delta_r_squared_imputed"],
    "imputed_event_count": result["imputed_event_count"],
    "notes": RUN_NOTES,
}

is_new = not LOG_PATH.exists()
with LOG_PATH.open("a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(row.keys()))
    if is_new:
        writer.writeheader()
    writer.writerow(row)

history = pd.read_csv(LOG_PATH)
print(f"Logged run #{len(history)} to {LOG_PATH.name}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.index, history["delta_r_squared"], marker="o", label="delta_r_squared (ungefuellt)")
ax.plot(history.index, history["delta_r_squared_imputed"], marker="o", label="delta_r_squared_imputed (contest metric)")
ax.axhline(0.049, color="gray", linestyle="--", linewidth=1, label="Gemini baseline (0.049)")
ax.set_xlabel("run #")
ax.set_ylabel("delta R^2")
ax.set_title("Delta R^2 over time")
ax.legend()
plt.tight_layout()
plt.show()

history.tail(10)

## Where to go from here

The archive gives you the **inputs** — events and their disclosures. To turn this
into a backtest you'd:

1. Pair each event with the realized next-day reaction for its focal asset, from
   your own market-data source.
2. Convert that reaction into a percentile rank in `[0, 1]` **across the
   quarter's cross-section** of scored event outcomes — a rank against all the
   period's events, not against the asset's own history. That's the target the
   competition scores; `examples.scoring.percentile_ranks` is the exact
   transform (see section 5).
3. Score your model's predicted percentile against it offline, and iterate.

When your model is ready, move it into a deployed webhook handler (see the starter
repos) — that's what receives live events and submits predictions. These notebooks
deliberately stop short of `POST /predictions`.